# SCE-Net для задачи совместимости пар одежды (image-only)

Этот ноутбук реализует **Similarity Condition Embedding Network (SCE-Net)** из статьи *Learning Similarity Conditions Without Explicit Supervision* для вашей постановки:
- есть пары `sku1`, `sku2`;
- метка `target in {good, bad}`;
- пути к изображениям `sku1_path`, `sku2_path`.

Мы используем **только визуальную модальность** и берём энкодер признаков из `patrickjohncyh/fashion-clip` (Hugging Face), как вы и попросили.

---

## Как это связано со статьёй

1. **Общий эмбеддинг изображения `V`** (в статье это выход CNN): у нас это выход **FashionCLIP image encoder**.
2. **Similarity condition masks `C_1 ... C_M`** (Раздел 3.1, Eq. 1): обучаемый параметр `[M, D]`, применяем `E_ij = C_j ⊙ V_i`.
3. **Condition weight branch** (Раздел 3.2, Eq. 3): вход `[V_i, V_j]`, MLP + ReLU + Softmax, выход `w` размера `M`.
4. **Агрегация** (Eq. 2): `E_i = w @ O_i`, где `O_i` — masked-векторы `[M, D]`.
5. **Triplet loss** (Eq. 4).
6. **Регуляризация L1/L2** (Eq. 5).

Важно: у вас парные метки good/bad, а triplet требует триплеты. Поэтому ниже строим триплеты из парного датасета.

## 1) Установка и импорты

> Если уже работаете в окружении с `torch/transformers`, этот блок можно пропустить.

In [ ]:
# !pip install -q torch torchvision transformers pandas scikit-learn pillow tqdm matplotlib

import os
import random
from dataclasses import dataclass
from typing import Dict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import CLIPModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2) Конфигурация

- `M` — число similarity conditions (в статье это гиперпараметр; в абляции лучшее около 5).
- `D` берём автоматически из FashionCLIP.

In [ ]:
@dataclass
class Config:
    csv_path: str = 'data/pairs.csv'
    image_root: str = ''

    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1

    batch_size: int = 64
    num_workers: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-5
    epochs: int = 8
    margin: float = 0.2
    lambda_l1: float = 1e-5
    lambda_l2: float = 1e-4

    freeze_backbone: bool = True
    train_size: float = 0.8
    val_size: float = 0.1
    test_size: float = 0.1

cfg = Config()
cfg

## 3) Загрузка и валидация данных

Ожидаемый CSV: `sku1, sku2, target, sku1_path, sku2_path`.
Преобразуем таргет: `good -> 1`, `bad -> 0`.

In [ ]:
def normalize_target(x: str) -> int:
    x = str(x).strip().lower()
    if x in {'good', '1', 'true', 'yes', 'compatible'}:
        return 1
    if x in {'bad', '0', 'false', 'no', 'incompatible'}:
        return 0
    raise ValueError(f'Unknown target value: {x}')


def full_path(p: str, root: str = '') -> str:
    p = str(p)
    if root and not os.path.isabs(p):
        return os.path.join(root, p)
    return p


df = pd.read_csv(cfg.csv_path)
required_cols = {'sku1', 'sku2', 'target', 'sku1_path', 'sku2_path'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['y'] = df['target'].apply(normalize_target)
df['sku1_path'] = df['sku1_path'].apply(lambda p: full_path(p, cfg.image_root))
df['sku2_path'] = df['sku2_path'].apply(lambda p: full_path(p, cfg.image_root))

mask_exists = df['sku1_path'].map(os.path.exists) & df['sku2_path'].map(os.path.exists)
missing_rows = (~mask_exists).sum()
if missing_rows > 0:
    print(f'WARNING: dropped {missing_rows} rows with missing image files')
df = df[mask_exists].reset_index(drop=True)

print(df.shape)
print(df['y'].value_counts(normalize=True))
df.head()

## 4) Split train/val/test

Стратифицированный split по `y`.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=(1.0 - cfg.train_size),
    stratify=df['y'],
    random_state=SEED,
)

val_ratio_in_temp = cfg.val_size / (cfg.val_size + cfg.test_size)
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_ratio_in_temp),
    stratify=temp_df['y'],
    random_state=SEED,
)

print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)

## 5) Построение триплетов из парного датасета

Статья обучает SCE-Net через triplet loss. Для anchor `a` берём:
- `p` из good-пар с `a`;
- `n` из bad-пар с `a`.

Так формируем `(a,p,n)` без явных condition labels — в духе статьи.

In [ ]:
def build_affinity_maps(pairs_df: pd.DataFrame):
    pos_map: Dict[str, set] = {}
    neg_map: Dict[str, set] = {}
    path_map: Dict[str, str] = {}

    for row in pairs_df.itertuples(index=False):
        a, b, y = str(row.sku1), str(row.sku2), int(row.y)
        pa, pb = str(row.sku1_path), str(row.sku2_path)
        path_map[a] = pa
        path_map[b] = pb

        if y == 1:
            pos_map.setdefault(a, set()).add(b)
            pos_map.setdefault(b, set()).add(a)
        else:
            neg_map.setdefault(a, set()).add(b)
            neg_map.setdefault(b, set()).add(a)

    return pos_map, neg_map, path_map


def build_triplets(pairs_df: pd.DataFrame, max_triplets_per_anchor: int = 50):
    pos_map, neg_map, path_map = build_affinity_maps(pairs_df)
    triplets = []
    anchors = sorted(set(list(pos_map.keys()) + list(neg_map.keys())))
    rng = random.Random(SEED)

    for a in anchors:
        pos = list(pos_map.get(a, []))
        neg = list(neg_map.get(a, []))
        if len(pos) == 0 or len(neg) == 0:
            continue

        for _ in range(min(max_triplets_per_anchor, len(pos) * len(neg))):
            p = rng.choice(pos)
            n = rng.choice(neg)
            triplets.append((a, p, n, path_map[a], path_map[p], path_map[n]))

    return pd.DataFrame(triplets, columns=[
        'anchor_sku','pos_sku','neg_sku','anchor_path','pos_path','neg_path'
    ])


train_triplets = build_triplets(train_df, max_triplets_per_anchor=40)
val_triplets = build_triplets(val_df, max_triplets_per_anchor=20)

print('train_triplets:', train_triplets.shape)
print('val_triplets:', val_triplets.shape)
train_triplets.head()

## 6) Dataset/DataLoader (подробно)

### Зачем нужен `make_triplet_collate`
В `TripletImageDataset` мы возвращаем **три PIL-изображения** (`anchor`, `positive`, `negative`) на каждый индекс.
`DataLoader` по умолчанию не знает, как эффективно собрать такие объекты в батч для CLIP. Поэтому мы задаём свой `collate_fn`:

1. получает список элементов батча `[(a1,p1,n1), (a2,p2,n2), ...]`;
2. раскладывает его на 3 списка картинок;
3. применяет `CLIPProcessor(..., return_tensors='pt')` отдельно для anchor/positive/negative;
4. возвращает уже готовые `pixel_values` тензоры для модели.

### Зачем в `TripletImageDataset` передаётся `processor`
Сейчас `processor` не используется внутри `__getitem__` (препроцессинг делается в `collate_fn`), но хранится в объекте как:
- удобство для будущего расширения (например, аугментации прямо в dataset),
- явная связь dataset с конкретным image-preprocessor,
- более читаемый API (`TripletImageDataset(..., processor=...)`).


In [ ]:
class TripletImageDataset(Dataset):
    def __init__(self, triplet_df: pd.DataFrame, processor: CLIPProcessor):
        # Сохраняем таблицу с триплетами и сбрасываем индекс для стабильного __getitem__
        self.df = triplet_df.reset_index(drop=True)
        # processor сохраняем для явной связки dataset<->preprocess pipeline
        # (даже если в текущей версии токенизация/resize делается в collate_fn)
        self.processor = processor

    def __len__(self):
        # Количество триплетов
        return len(self.df)

    def _load_img(self, p: str) -> Image.Image:
        # Читаем изображение и приводим к RGB (важно: CLIP ожидает 3 канала)
        return Image.open(p).convert('RGB')

    def __getitem__(self, idx):
        # Берём одну строку-триплет
        row = self.df.iloc[idx]
        # Возвращаем три PIL-изображения: anchor/positive/negative
        return self._load_img(row['anchor_path']), self._load_img(row['pos_path']), self._load_img(row['neg_path'])


def make_triplet_collate(processor: CLIPProcessor):
    """
    Возвращает функцию collate_fn для DataLoader.

    Что делает collate_fn:
    1) Получает список из элементов батча: [(a,p,n), ...]
    2) Разбивает на 3 списка изображений
    3) Батчево прогоняет через CLIPProcessor
    4) Возвращает 3 тензора pixel_values формы [B, 3, H, W]
    """
    def collate_fn(batch):
        # unzip списка триплетов по осям
        a_imgs, p_imgs, n_imgs = zip(*batch)

        # Отдельно процессим anchor/positive/negative,
        # чтобы дальше удобно считать две пары: (a,p) и (a,n)
        a_inputs = processor(images=list(a_imgs), return_tensors='pt')
        p_inputs = processor(images=list(p_imgs), return_tensors='pt')
        n_inputs = processor(images=list(n_imgs), return_tensors='pt')

        # Возвращаем только pixel_values, т.к. используем image-only
        return a_inputs['pixel_values'], p_inputs['pixel_values'], n_inputs['pixel_values']

    return collate_fn

## 7) Реализация SCE-Net (формулы Eq.1–Eq.3) — подробный разбор

Ниже класс `SCENet` почти строка-в-строку повторяет идею статьи:

- **Eq.1**: `E_ij = C_j ⊙ V_i`
  - `C_j` — j-я mask (обучаемый параметр)
  - `V_i` — общий эмбеддинг изображения
- **Eq.3**: `w = softmax(MLP(concat(V_i, V_j)))`
  - branch вычисляет релевантность каждого условия под конкретную пару
- **Eq.2**: `E_i = w @ O_i`
  - `O_i` — набор masked-эмбеддингов `[M, D]`

Ключевая мысль: сравнение двух айтемов выполняется в **динамически выбранном semantic subspace**.


In [ ]:
class SCENet(nn.Module):
    def __init__(self, clip_model_name: str, num_conditions: int = 5, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        # 1) Базовый image encoder (вместо CNN из статьи используем FashionCLIP)
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        # 2) Определяем D — размер общего image embedding пространства
        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            emb_dim = proj.out_features if hasattr(proj, 'out_features') else self.clip.config.projection_dim
        else:
            emb_dim = getattr(self.clip.config, 'projection_dim', None)
            if emb_dim is None:
                raise ValueError('Cannot infer embedding dimension D from model config/projection.')

        self.emb_dim = emb_dim
        self.num_conditions = num_conditions

        # 3) Similarity condition masks C in R^{M x D} (Eq.1)
        self.condition_masks = nn.Parameter(torch.empty(num_conditions, emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)

        # 4) Condition weight branch (Eq.3): concat(Vi,Vj) -> softmax weights w in R^M
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions)
        )

    def _extract_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Унифицированно извлекает Tensor [B, D] из разных форматов выхода HF."""
        if hasattr(self.clip, 'get_image_features'):
            out = self.clip.get_image_features(pixel_values=pixel_values)
            if torch.is_tensor(out):
                return out
            if hasattr(out, 'image_embeds') and out.image_embeds is not None:
                return out.image_embeds
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                return out.pooler_output
            if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
                return out.last_hidden_state[:, 0, :]

        out = self.clip(pixel_values=pixel_values, return_dict=True)
        if hasattr(out, 'image_embeds') and out.image_embeds is not None:
            return out.image_embeds
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            return out.last_hidden_state[:, 0, :]

        raise TypeError(f'Unsupported image output type: {type(out)}')

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        1) Извлекаем image features из backbone
        2) L2-нормализуем по последней оси (dim=-1)

        Зачем нормализация:
        - стабилизирует масштаб признаков;
        - делает расстояния между векторами более устойчивыми;
        - полезно для metric learning / triplet objective.
        """
        img_feat = self._extract_image_features(pixel_values)
        return F.normalize(img_feat, p=2, dim=-1)

    def compute_w(self, v1: torch.Tensor, v2: torch.Tensor) -> torch.Tensor:
        # Eq.3: w = softmax(MLP(concat(Vi, Vj)))
        logits = self.weight_branch(torch.cat([v1, v2], dim=-1))
        return F.softmax(logits, dim=-1)

    def apply_conditions(self, v: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        # Eq.1: O_i[j] = C_j ⊙ V_i
        masked = v.unsqueeze(1) * self.condition_masks.unsqueeze(0)  # [B,M,D]
        # Eq.2: E_i = w @ O_i
        return torch.einsum('bm,bmd->bd', w, masked)

    def forward_pair(self, img1: torch.Tensor, img2: torch.Tensor):
        # Получаем общие эмбеддинги пары
        v1 = self.encode_image(img1)
        v2 = self.encode_image(img2)

        # Считаем веса условий именно для этой пары
        w = self.compute_w(v1, v2)

        # Проецируем оба объекта в релевантный subspace
        e1 = self.apply_conditions(v1, w)
        e2 = self.apply_conditions(v2, w)
        return e1, e2, w, v1, v2

## 8) Loss (Eq.4 + Eq.5) — подробный разбор

Функция `sce_loss` реализует objective из статьи:

\[
L = L_{triplet} + \lambda_1 \cdot L_1(C) + \lambda_2 \cdot L_2(E)
\]

Где:
- `d_pos = ||E_a - E_p||_2` — расстояние anchor до positive;
- `d_neg = ||E_a - E_n||_2` — расстояние anchor до negative;
- `triplet = max(0, d_pos - d_neg + margin)` (Eq.4).

Почему именно такие строки:
- `torch.norm(..., p=2, dim=-1)` — это евклидова дистанция по embedding-компонентам;
- нам нужно, чтобы `d_pos < d_neg` хотя бы на `margin`.

Если условие уже выполнено, вклад триплета 0; если нет — модель получает штраф.


In [ ]:
def sce_loss(e_anchor, e_pos, e_neg, condition_masks, margin, lambda_l1, lambda_l2):
    # Евклидовы расстояния (Eq.4)
    # d_pos: насколько anchor близок к positive (хотим МЕНЬШЕ)
    d_pos = torch.norm(e_anchor - e_pos, p=2, dim=-1)
    # d_neg: насколько anchor далёк от negative (хотим БОЛЬШЕ)
    d_neg = torch.norm(e_anchor - e_neg, p=2, dim=-1)

    # Triplet hinge: max(0, d_pos - d_neg + margin)
    # Если d_neg >= d_pos + margin, ошибка по этому примеру = 0
    triplet = F.relu(d_pos - d_neg + margin).mean()

    # L1 на masks C: по статье стимулирует sparsity/disentanglement условий
    l1 = condition_masks.abs().mean()

    # L2 на репрезентации: регуляризуем масштаб embedding-ов
    l2 = (e_anchor.pow(2).mean() + e_pos.pow(2).mean() + e_neg.pow(2).mean()) / 3.0

    # Финальная функция (Eq.5)
    total = triplet + lambda_l1 * l1 + lambda_l2 * l2

    # Возвращаем и loss tensor, и удобные scalar-метрики для логирования
    return total, {'loss': total.item(), 'triplet': triplet.item(), 'l1': l1.item(), 'l2': l2.item()}

## 9) Инициализация обучения

In [ ]:
processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)

train_ds = TripletImageDataset(train_triplets, processor)
val_ds = TripletImageDataset(val_triplets, processor)
collate_fn = make_triplet_collate(processor)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)

model = SCENet(cfg.hf_model_name, cfg.num_conditions, cfg.condition_hidden, cfg.dropout).to(device)

if cfg.freeze_backbone:
    for p in model.clip.parameters():
        p.requires_grad = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('embedding dim D:', model.emb_dim)

## 10) Train/Val loop (подробно + ROC-AUC на триплетах)

Что делает `run_epoch`:
1. Переводит модель в `train` или `eval` режим.
2. Для каждого батча берёт две пары: `(anchor, positive)` и `(anchor, negative)`.
3. Для каждой пары считает свои condition weights `w` (по идее статьи branch зависит от пары).
4. Собирает итоговый `sce_loss` и, если train, делает backward + optimizer step.
5. Дополнительно считает **proxy ROC-AUC** прямо на триплетах:
   - пара `(a,p)` считается классом 1,
   - пара `(a,n)` считается классом 0,
   - score = `-||E_a - E_x||`.

Важно: этот AUC полезен как онлайн-индикатор, но «чистый» продуктовый AUC лучше считать на исходных pair-данных (`pair_scores` ниже).


In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    losses, triplets = [], []
    # Для proxy ROC-AUC из триплетов
    y_true, y_score = [], []

    for a_px, p_px, n_px in tqdm(loader, leave=False):
        # Перенос батча на устройство
        a_px, p_px, n_px = a_px.to(device), p_px.to(device), n_px.to(device)

        with torch.set_grad_enabled(is_train):
            # Ветка (anchor, positive)
            e_a_pos, e_p, _, _, _ = model.forward_pair(a_px, p_px)
            # Ветка (anchor, negative)
            e_a_neg, e_n, _, _, _ = model.forward_pair(a_px, n_px)

            # Anchor в двух контекстах пар может слегка отличаться; усредняем
            e_a = 0.5 * (e_a_pos + e_a_neg)

            # Основной loss из статьи (Eq.4 + Eq.5)
            loss, stats = sce_loss(e_a, e_p, e_n, model.condition_masks, cfg.margin, cfg.lambda_l1, cfg.lambda_l2)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        losses.append(stats['loss'])
        triplets.append(stats['triplet'])

        # --- Proxy классификационный сигнал из триплета ---
        # good-пара: (anchor, positive) -> label 1
        ap_score = (-torch.norm(e_a - e_p, p=2, dim=-1)).detach().cpu().numpy()
        # bad-пара: (anchor, negative) -> label 0
        an_score = (-torch.norm(e_a - e_n, p=2, dim=-1)).detach().cpu().numpy()

        y_score.extend(ap_score.tolist())
        y_score.extend(an_score.tolist())
        y_true.extend([1] * len(ap_score))
        y_true.extend([0] * len(an_score))

    out = {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'triplet': float(np.mean(triplets)) if triplets else np.nan,
    }

    if len(set(y_true)) == 2:
        out['triplet_pair_auc'] = float(roc_auc_score(y_true, y_score))
    else:
        out['triplet_pair_auc'] = np.nan

    return out


best_val = float('inf')
best_path = 'sce_net_best.pt'
history = []

for epoch in range(1, cfg.epochs + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | val_loss={val_metrics['loss']:.4f} | "
        f"train_triplet_auc={train_metrics['triplet_pair_auc']:.4f} | "
        f"val_triplet_auc={val_metrics['triplet_pair_auc']:.4f}"
    )

    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        torch.save({'model_state': model.state_dict(), 'config': cfg.__dict__}, best_path)
        print('  -> saved best checkpoint:', best_path)

pd.DataFrame(history)

## 11) Pair scoring и метрики

На инференсе для пары считаем `score = -||E_i - E_j||` и оцениваем AUC/PR-AUC.

In [ ]:
@torch.no_grad()
def pair_scores(model: SCENet, pair_df: pd.DataFrame, processor: CLIPProcessor, batch_size: int = 64):
    model.eval()
    scores = []
    labels = pair_df['y'].astype(int).tolist()

    rows = pair_df[['sku1_path', 'sku2_path']].values.tolist()
    for i in tqdm(range(0, len(rows), batch_size)):
        chunk = rows[i:i+batch_size]
        imgs1 = [Image.open(a).convert('RGB') for a, _ in chunk]
        imgs2 = [Image.open(b).convert('RGB') for _, b in chunk]

        px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
        px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)

        e1, e2, _, _, _ = model.forward_pair(px1, px2)
        scores.extend((-torch.norm(e1 - e2, p=2, dim=-1)).cpu().numpy().tolist())

    return np.array(scores), np.array(labels)


ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state'])

val_scores, val_labels = pair_scores(model, val_df, processor)
print('VAL ROC-AUC=', roc_auc_score(val_labels, val_scores), 'PR-AUC=', average_precision_score(val_labels, val_scores))

test_scores, test_labels = pair_scores(model, test_df, processor)
print('TEST ROC-AUC=', roc_auc_score(test_labels, test_scores), 'PR-AUC=', average_precision_score(test_labels, test_scores))

## 12) Интерпретация conditions

Можно посмотреть, как распределяются веса `w` по условиям для конкретной пары.

In [ ]:
@torch.no_grad()
def inspect_pair_weights(model: SCENet, processor: CLIPProcessor, img_path_1: str, img_path_2: str):
    model.eval()
    img1 = Image.open(img_path_1).convert('RGB')
    img2 = Image.open(img_path_2).convert('RGB')
    px1 = processor(images=[img1], return_tensors='pt')['pixel_values'].to(device)
    px2 = processor(images=[img2], return_tensors='pt')['pixel_values'].to(device)

    _, _, w, _, _ = model.forward_pair(px1, px2)
    w = w[0].cpu().numpy()
    for i, wi in enumerate(w, 1):
        print(f'C{i}: {wi:.4f}')
    return w

# sample = test_df.iloc[0]
# inspect_pair_weights(model, processor, sample['sku1_path'], sample['sku2_path'])

## FAQ по коду (ответы на ваши вопросы)

### Что делает `def make_triplet_collate`?
Это фабрика `collate_fn` для `DataLoader`: она собирает список триплетов в батч и прогоняет изображения через `CLIPProcessor`, чтобы модель получила тензоры `pixel_values`.

### Для чего в `TripletImageDataset` передается `self.processor = processor`?
Для явной конфигурации dataset и для будущего расширения пайплайна (аугментации/процессинг в `__getitem__`). В текущем варианте основная обработка в `collate_fn`.

### Раскрой каждую строчку в классе `SCENet`
Сделано комментариями прямо внутри класса: инициализация backbone, вычисление D, masks `C`, weight-branch `w`, Eq.1/2/3 и `forward_pair`.

### Что делает `encode_image(...)`?
Извлекает визуальный вектор и L2-нормализует его. Нормализация стабилизирует масштаб эмбеддингов и расстояния для metric-learning.

### Что происходит в `sce_loss(...)`?
Считаем `d_pos`/`d_neg`, triplet hinge (Eq.4), добавляем L1/L2 (Eq.5), суммируем в итоговый objective.

### Почему `d_pos = ||Ea-Ep||`, `d_neg = ||Ea-En||`?
Потому что это суть triplet metric learning: positive ближе к anchor, negative дальше.

### Что происходит в `run_epoch`?
Один проход по всем батчам: forward для `(a,p)` и `(a,n)`, loss, шаг оптимизатора (в train), сбор метрик. Дополнительно добавлен proxy ROC-AUC на триплет-парах.


## 13) Что дальше улучшать

1. Тюнинг `M in [1,2,5,10]` по валидации (как в абляциях статьи).
2. Добавить semi-hard/hard negative mining.
3. Для production кэшировать image embeddings `V` офлайн и онлайн считать только weight-branch + маскирование.

Реализация выше следует статье по ключевым формулам Eq.1-Eq.5, но адаптирована к вашим парным меткам через построение триплетов.